<a href="https://colab.research.google.com/github/Alex-Zeo/audio_downloader/blob/main/Audio_Downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Audio/Video Downloader - X Spaces, Youtube, Apple Podcasts, Spotify Podcasts
Downloads public videos, podcasts, playlists, or channels/authors
*   Input: CLI URL (non DRMM protected only for academic purposes or for personal transcription)
*   Output: wav file with the name of your URL
  
Dependencies:
  - yt_dlp, tqdm, pydub, ffmpeg
  - For Google Colab: google.colab.driv

In [ ]:
# Install required packages (if running in Colab):
# ---------------------------------------------------------
!pip install --quiet --upgrade yt_dlp tqdm pydub
!apt-get update && apt-get install -y ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.5/175.5 kB 636.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 9.4 MB/s eta 0:00:00
Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,923 kB]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease [24.3 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:11 https://ppa.launchpadcontent.net

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%%writefile audio_downloader.py
#!/usr/bin/env python3
"""
Audio / Video Downloader  – 2025-07-31  (Colab ready)
----------------------------------------------------
• For every embedded object in a tweet, YouTube link, etc.:
      └─ exports  ① loss-less WAV   ② MP4 (video-only OK)
• Handles tweet carousels (_01, _02 … filenames).
• Saves cleaned yt-dlp metadata → .json sidecar.
Default Colab output dir → /content/media_downloads
"""

from __future__ import annotations
import argparse, json, logging, logging.config, os, tempfile
from pathlib import Path
from typing import List, Dict, Any, Tuple

import yt_dlp
from yt_dlp.utils import DownloadError
from tqdm import tqdm

# ── Colab detection ────────────────────────────────────────────────
try:
    from google.colab import files            # type: ignore
    from IPython.display import HTML, display  # type: ignore
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

# ── JSON logger ────────────────────────────────────────────────────
LOG_CFG = {
    "version": 1,
    "handlers": {"c": {"class": "logging.StreamHandler", "formatter": "fmt"}},
    "formatters": {
        "fmt": {"format": '{"ts":"%(asctime)s","lvl":"%(levelname)s","mod":"%(name)s","line":%(lineno)d,"msg":"%(message)s"}'}
    },
    "root": {"handlers": ["c"], "level": "INFO"},
}
logging.config.dictConfig(LOG_CFG)
log = logging.getLogger("audio_dl")

# ── TQDM progress hook ─────────────────────────────────────────────
_pbar: tqdm | None = None
def _progress(h: Dict[str, Any]) -> None:
    global _pbar
    if h["status"] == "downloading":
        total = h.get("total_bytes") or h.get("total_bytes_estimate")
        if _pbar is None:
            _pbar = tqdm(total=total, unit="B", unit_scale=True, desc="download")
        _pbar.update(h["downloaded_bytes"] - _pbar.n)
    elif h["status"] == "finished" and _pbar:
        _pbar.close(); _pbar = None

# ── Utilities ──────────────────────────────────────────────────────
def _safe_stem(url: str) -> str:
    stem = url.split("://")[-1].rstrip("/").split("?")[0]
    return stem.replace("/", "_")[:100]

def _save_meta(info: Dict[str, Any], stem: Path) -> None:
    fp = stem.with_suffix(".json")
    with fp.open("w", encoding="utf-8") as fh:
        json.dump({k: v for k, v in info.items() if not k.startswith("_")}, fh, indent=2)
    log.info(f"meta:{fp}")

# ── Format selector ────────────────────────────────────────────────
def _best_format(info: Dict[str, Any], want_video=True) -> str:
    has_video = any(f.get("video_ext") != "none" for f in info.get("formats", []))
    return "bestvideo*+bestaudio/best" if want_video and has_video else "bestaudio/best"

# ── Download a single media entry ──────────────────────────────────
def _dl_entry(
    info: Dict[str, Any],
    stem: Path,
    idx: int,
    keep_tmp: bool,
    want_video: bool,
) -> List[Path]:
    out: List[Path] = []
    fmts = _best_format(info, want_video)
    log.info(f"entry:{idx}  fmt:{fmts}")

    ydl_opts = {
        "format": fmts,
        "outtmpl": f"{stem}_{idx:02d}.%(ext)s",
        "progress_hooks": [_progress],
        "retries": 3,
        "merge_output_format": "mp4",
        "keepvideo": keep_tmp,
    }

    # yt-dlp expects plain str paths
    if keep_tmp:
        ydl_opts["paths"] = {"home": str(stem.parent)}

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.process_ie_result(info, download=True)
    except DownloadError as e:
        log.error(f"dl_err:{e}")
        return out

    for ext in ("mp4", "wav"):
        fp = Path(f"{stem}_{idx:02d}.{ext}")
        if fp.exists():
            out.append(fp)

    _save_meta(info, stem.with_name(f"{stem.name}_{idx:02d}"))
    return out

# ── Pull all embeds (tweet playlist etc.) ──────────────────────────
def _dl_url(url: str, out_dir: Path, keep_tmp: bool, audio_only: bool) -> Tuple[List[Path], List[str]]:
    # Probe once to get playlist + entries
    with yt_dlp.YoutubeDL({"skip_download": True}) as ydl:
        top_info = ydl.extract_info(url, download=False)
    entries = top_info.get("entries") or [top_info]

    files, failed = [], []
    for idx, entry in enumerate(entries, 1):
        files += _dl_entry(entry, out_dir / _safe_stem(url), idx, keep_tmp, not audio_only)
    if not files:
        failed.append(url)
    return files, failed

# ── Colab link helper ──────────────────────────────────────────────
def _link(p: Path) -> None:
    if not IS_COLAB: return
    try:
        files.download(str(p))
    except Exception:
        display(HTML(f'<a href="./files/{p.name}" download>{p.name}</a>'))

# ── CLI and main ───────────────────────────────────────────────────
def _parse():
    p = argparse.ArgumentParser(description="Dump WAV+MP4 from X / YouTube / Instagram.")
    p.add_argument("--urls", nargs="+", required=True)
    p.add_argument("--output_dir", default="media_downloads")
    p.add_argument("--keep", action="store_true")
    p.add_argument("--audio_only", action="store_true")
    return p.parse_args()

def main() -> None:
    a = _parse()
    out_root = Path(a.output_dir)
    out_root.mkdir(parents=True, exist_ok=True)

    log.info(f"target_dir:{out_root}")
    log.info(f"total_urls:{len(a.urls)}")

    ok, bad = [], []
    for u in a.urls:
        f, b = _dl_url(u, out_root, keep_tmp=a.keep, audio_only=a.audio_only)
        ok += f; bad += b

    log.info(f"done_ok:{len(ok)} done_fail:{len(bad)}")
    for p in ok:
        _link(p); meta = p.with_suffix(".json"); _link(meta)
    if bad:
        log.warning(f"failed:{bad}")

if __name__ == "__main__":
    main()


Writing audio_downloader.py


In [ ]:
!python audio_downloader.py \
    --urls "https://x.com/i/spaces/1ZkKzYZrvgdxv" --keep

{"ts":"2025-08-11 13:35:11,840","lvl":"INFO","mod":"audio_dl","line":145,"msg":"target_dir:media_downloads"}
{"ts":"2025-08-11 13:35:11,840","lvl":"INFO","mod":"audio_dl","line":146,"msg":"total_urls:1"}
[twitter:spaces] Extracting URL: https://x.com/i/spaces/1ZkKzYZrvgdxv
[twitter:spaces] 1ZkKzYZrvgdxv: Downloading guest token
[twitter:spaces] 1ZkKzYZrvgdxv: Downloading GraphQL JSON
[twitter:spaces] 28_1954361601345392640: Downloading guest token
[twitter:spaces] 28_1954361601345392640: Downloading legacy API JSON
[twitter:spaces] 28_1954361601345392640: Downloading m3u8 information
{"ts":"2025-08-11 13:35:15,800","lvl":"INFO","mod":"audio_dl","line":79,"msg":"entry:1  fmt:bestaudio/best"}
[info] 1ZkKzYZrvgdxv: Downloading 1 format(s): 0
[download] Destination: media_downloads/media_downloads/x.com_i_spaces_1ZkKzYZrvgdxv_01.m4a
[hls @ 0x59389eaa7100] Skip ('#EXT-X-VERSION:6')
[hls @ 0x59389eaa7100] Skip ('#EXT-X-INDEPENDENT-SEGMENTS')
[hls @ 0x59389eaa7100] Skip ('#EXT-X-PROGRAM-DATE-